# 27 - Persistent Homology Refined: Zigzag, Levelsets & Instability Bands

Standard persistence works for monotone filtrations $K_0 \subset K_1 \subset \cdots$. But real data has non-monotone parameter paths: temperature can go up and then down, a time-varying complex can grow and shrink. **Zigzag persistence** (Carlsson-de Silva 2010) handles exactly this case: a sequence
$$K_0 \leftrightarrow K_1 \leftrightarrow K_2 \leftrightarrow \cdots$$
where each arrow is an inclusion going either way. The resulting barcodes track topological features through non-monotone parameter paths.

## Learning Goals
- Use `compute_barcodes_exact` for certified barcodes with `exact=True`
- Use `compute_zigzag_persistence` for non-monotone filtration sequences
- Use `compute_topological_loss` for topology-guided optimization
- Understand levelset persistence trees for real-valued functions
- Build zigzag complexes from point cloud sequences with `build_union_intersection_zigzag`

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | Point cloud sequence or levelset filtration | `SimplicialComplex` list |
| **Algebraic** | Barcode diagram = multiset of intervals $[b, d)$ | `BarcodeResult` |
| **Result** | Certified barcodes with `exact=True` and `theorem_tag` | `compute_barcodes_exact` |

## Formal Background

For a zigzag sequence $K_0 \leftrightarrow K_1 \leftrightarrow \cdots \leftrightarrow K_n$, the zigzag persistence theorem (Gabriel's theorem applied to quiver representations) guarantees a decomposition into interval modules $[b, d]$. The resulting barcode is unique up to isomorphism.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pysurgery.core.persistent_homology import (
    compute_barcodes_exact, BarcodeResult, Barcode,
    compute_zigzag_persistence,
    compute_topological_loss,
)
from pysurgery.core.temporal_topology import build_union_intersection_zigzag
from pysurgery.core.complexes import SimplicialComplex

print("=" * 70)
print("27 - Persistent Homology Refined: Setup Complete")
print("=" * 70)

## Part 1: Exact Barcodes — `compute_barcodes_exact`

Standard TDA libraries (Ripser, Gudhi) compute barcodes with floating-point arithmetic, which can introduce rounding errors for degenerate filtrations. `compute_barcodes_exact` uses exact integer/rational arithmetic internally:
- **Field `'Z2'`**: mod-2 coefficients (fast, loses orientation information)
- **Field `'Q'`**: rational coefficients (exact, detects orientability)

The `exact=True` flag on the result certifies that the barcode was computed over the chosen field with no floating-point rounding.

In [ ]:
# Filtered S² with 3 levels: vertex → edges+vertex → full
S2 = SimplicialComplex.sphere(2)
filtration = S2.filtered(levels=[0.0, 0.5, 1.0])

# Exact barcodes in degree 1 over Z/2
result_Z2: BarcodeResult = compute_barcodes_exact(
    filtration, dimension=1, field="Z2", backend="auto"
)
print("H₁ barcodes of S² filtration (Z/2 coefficients):")
for b in result_Z2.barcodes:
    print(f"  [{b.birth:.2f}, {b.death:.2f})")
print(f"  exact:        {result_Z2.exact}")
print(f"  backend_used: {result_Z2.backend_used}")
print(f"  theorem_tag:  {result_Z2.theorem_tag}")

print()
# Over Q
result_Q: BarcodeResult = compute_barcodes_exact(
    filtration, dimension=1, field="Q", backend="auto"
)
print("H₁ barcodes of S² filtration (Q coefficients):")
for b in result_Q.barcodes:
    print(f"  [{b.birth:.2f}, {b.death:.2f})")

print()
# H₂ (the fundamental class appears at level 1.0)
result_H2: BarcodeResult = compute_barcodes_exact(
    filtration, dimension=2, field="Q", backend="auto"
)
print("H₂ barcodes (the sphere itself):")
for b in result_H2.barcodes:
    print(f"  [{b.birth:.2f}, {b.death:.2f})")

## Part 2: Building Zigzag Complexes from Point Clouds

`build_union_intersection_zigzag` constructs the canonical zigzag sequence from a list of point clouds:
$$K_0 \hookrightarrow K_0 \cup K_1 \hookleftarrow K_1 \hookrightarrow K_1 \cup K_2 \hookleftarrow K_2 \cdots$$
This is the union-intersection zigzag. It tracks which topological features persist across time steps.

In [ ]:
rng = np.random.default_rng(0)

# Synthetic time series: point cloud morphs from 1 ring to 2 rings
n_steps = 8
point_clouds = []
for t in range(n_steps):
    if t < 4:
        # Single loop
        angles = np.linspace(0, 2*np.pi, 25)
        pts = np.column_stack([np.cos(angles), np.sin(angles)]) + rng.normal(0, 0.08, (25, 2))
    else:
        # Two loops
        a = np.linspace(0, 2*np.pi, 12)
        pts = np.vstack([
            np.column_stack([np.cos(a) - 1.2, np.sin(a)]) + rng.normal(0, 0.08, (12, 2)),
            np.column_stack([np.cos(a) + 1.2, np.sin(a)]) + rng.normal(0, 0.08, (12, 2)),
        ])
    point_clouds.append(pts)

# Build zigzag complex sequence
complex_seq = build_union_intersection_zigzag(
    point_clouds, epsilon=0.5, max_dimension=2
)
print(f"Zigzag sequence length: {len(complex_seq)}")
for i, K in enumerate(complex_seq):
    print(f"  K_{i}: {K.num_simplices(0)} vertices, {K.num_simplices(1)} edges")

## Part 3: Zigzag Persistence

`compute_zigzag_persistence` decomposes the zigzag module into interval modules. The resulting barcodes show when each topological feature (connected component, loop, void) first appears and last persists across the non-monotone sequence.

In [ ]:
zz_H1: BarcodeResult = compute_zigzag_persistence(complex_seq, field="Z2")

print("Zigzag H₁ barcodes:")
print("  (time steps 0-3: 1 loop, steps 4-7: 2 loops)")
print()
for b in zz_H1.barcodes:
    bar = "█" * int((b.death - b.birth) * 3) if b.death < 999 else "█" * 12 + "→"
    print(f"  [{b.birth:.0f}, {b.death:.0f})  {bar}")
print(f"  exact: {zz_H1.exact}")

# Visualise
fig, ax = plt.subplots(figsize=(9, 4))
for i, b in enumerate(zz_H1.barcodes):
    death = min(b.death, n_steps)
    ax.barh(i, death - b.birth, left=b.birth, height=0.6,
            color="steelblue", edgecolor="k", alpha=0.8)
ax.axvline(3.5, color="crimson", ls="--", lw=2, label="Phase transition")
ax.set_xlabel("Time step"); ax.set_ylabel("Feature index")
ax.set_title("Zigzag H₁ Barcodes: 1 Loop → 2 Loops Phase Transition")
ax.legend(); ax.set_xlim(0, n_steps)
plt.tight_layout(); plt.show()

## Part 4: Topological Loss for Optimization

`compute_topological_loss` measures the bottleneck distance between a barcode and a *target* barcode. This is differentiable (in an appropriate sense) and can be used in topology-guided optimization: force a point cloud to have a specific barcode by minimising the topological loss.

In [ ]:
# Define a target: exactly 1 loop of persistence ≥ 0.5
target_barcodes = [Barcode(birth=0.0, death=1.0)]

# Compute loss for each time step
losses = []
for t in range(n_steps):
    filtration_t = SimplicialComplex.rips(point_clouds[t], max_edge=0.6)
    result_t = compute_barcodes_exact(filtration_t, dimension=1, field="Z2")
    loss = compute_topological_loss(result_t.barcodes, target_barcodes, epsilon=0.1)
    losses.append(loss)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(n_steps), losses, "o-", color="steelblue", lw=2)
ax.axvline(3.5, color="crimson", ls="--", label="Phase transition")
ax.set_xlabel("Time step"); ax.set_ylabel("Topological loss")
ax.set_title("Topological Loss vs Target (1 loop)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Losses: {[f'{l:.3f}' for l in losses]}")
print("Steps 0-3: low loss (1 loop matches target)")
print("Steps 4-7: high loss (2 loops deviate from target)")

## Summary Checklist

- [x] Used `compute_barcodes_exact` with `field='Z2'` and `field='Q'` for certified barcodes
- [x] Built zigzag complex sequences with `build_union_intersection_zigzag`
- [x] Ran `compute_zigzag_persistence` to get barcodes for non-monotone filtrations
- [x] Detected a topological phase transition in the zigzag barcodes
- [x] Used `compute_topological_loss` to measure distance from a target barcode

## Next Steps
- **NB 33**: Temporal TDA extends this to full phase transition detection and bifurcation analysis
- **NB 34**: The capstone uses zigzag persistence for time-varying data inputs